# Marker Repo - Create and Submit Marker Lists

This notebook facilitates the creation and upload of a marker list.

**Components of a Marker List**
- **Metadata**: Entered manually, this part includes essential details about the marker list.
- **Markers**: Derived from a tab-delimited file with one or two columns.

**Process Workflow**
1. **Settings**: Specify the path to the marker file and details about its columns.
2. **Preparation**: The [Whitelist Repository](https://gitlab.gwdg.de/loosolab/software/metadata_whitelists) is downloaded or updated beforehand and is used to verify and extend the markers.
3. **Metadata Input**: Enter the first set of metadata such as the list's name, organism, and marker type (genes or genomic regions).
4. **Marker Processing**: If the markers are genes, they are filtered and expanded using whitelists (Gene name and Ensembl ID).
5. **Completing Metadata**: Enter the remaining metadata details using the [Metadata Organizer](https://gitlab.gwdg.de/loosolab/software/metadata-organizer).
6. **Saving the List**: Save the newly created marker list as yaml file.
7. **Final Validation**: Perform a last validation of the marker list.
8. **Publication**: Optionally, publish the list and add it to the repository.

## 1. Loading packages

In [ ]:
import markerrepo.marker_repo as mr
import markerrepo.parsing as parse
import markerrepo.update_uids as u_uids
import markerrepo.generate_metafile as gm
import markerrepo.validate_yaml as validate
import markerrepo.utils as utils

%load_ext autoreload
%autoreload 2

## 2. Settings

Specify path of the cloned repository.

In [ ]:
repo_path = "/mnt/workspace/mkessle/projects/annotate_by_marker_and_features"

The path of the list to be added to the marker repo. <br>
The file must consist of one marker per line or two tab separated columns (marker and info like cell type).

In [ ]:
LIST_PATH = '/mnt/workspace/mkessle/projects/annotate_by_marker_and_features/notebooks/test'

Additional information of the columns. <br>
marker_col: The column where the markers are stored <br>
info_col: The column where the information of the markers are stored (e.g. cell type, phase, ...). <br>
0 - first column, 1 - second column

If there are only markers available (one column), enter a string with the description of the markers into the info_col parameter - e.g. info_col = "mitochondrial".

In [ ]:
marker_col = 0
info_col = 1

## 3. Get whitelists

Pull whitelist repository and update if necessary.

In [ ]:
utils.get_whitelists(repo_path=repo_path)

## 4. Get essential metadata

Enter essential metadata: Liste name, organism and marker type

In [ ]:
LIST_NAME = input("Please enter the name of the marker list: ")
ORGANISM = mr.select(key="organism")
MARKER_TYPE = mr.select(key="marker_type")

## 5. Transform marker list

<b>Read</b>, <b>filter</b>, <b>extend</b> and <b>convert</b> marker list in order to append it to the yaml file.

In [ ]:
marker_list = mr.transform_marker_list(LIST_PATH, info_col, marker_col, MARKER_TYPE, ORGANISM)

## 6. Enter metadata

Enter general metadata, tags and add marker list(s) automatically.

In [ ]:
UID = mr.get_uid()
file_name = f"{LIST_NAME}_{UID}.yaml"

list_path = gm.generate_file(UID, LIST_NAME, False, marker_list, ORGANISM, MARKER_TYPE, repo_path=repo_path)

## 7. Validation

Check whether the format of the yaml file is correct.

In [ ]:
if validate.validate_file(utils.read_in_yaml(f"{list_path}", marker_list=False), repo_path=repo_path):
    print(f"No errors were found concerning the '{LIST_NAME}' marker list.")

## 8. Push list to repository

Pulls the latest changes, creates a new branch with the given list name, adds the new list, commits the changes and pushes the new branch to the remote repository.

In [ ]:
mr.push_marker_list(list_path, repo_path=repo_path)